# 🧠 AI Logic Engine: Main Inference Console

This notebook allows you to perform high-precision symbolic reasoning against your Firestore Knowledge Graph.

---

In [ ]:
# @title 🛠️ Step 1: Initialize Logic Engine
!pip install -q firebase-admin

from google.colab import files
import firebase_admin
from firebase_admin import credentials, firestore
import re

print("Please upload your serviceAccountKey.json file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2' 

if not firebase_admin._apps:
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Logic Engine Memory Initialized for Database: {DATABASE_ID}")

In [ ]:
# @title ⚙️ Step 2: Symbolic Inference Engine (Python Core)

class PythonLogicEngine:
    def __init__(self, db):
        self.db = db
        self.cache = {}

    def clean_id(self, text):
        return re.sub(r'[^a-z0-9]', '_', text.lower().strip())

    def get_node(self, node_id):
        key = self.clean_id(node_id)
        if not key: return None
        if key in self.cache: return self.cache[key]
        
        doc = self.db.collection('nodes').document(key).get()
        if doc.exists:
            data = doc.to_dict()
            self.cache[key] = data
            return data
        return None

    def find_path(self, start, end, max_depth=6):
        start_clean = self.clean_id(start)
        end_clean = self.clean_id(end)
        
        queue = [(start_clean, [], 1.0)]
        visited = {start_clean}
        
        while queue:
            node_key, path, certainty = queue.pop(0)
            
            if node_key == end_clean:
                return path, certainty
            
            if len(path) >= max_depth: continue
            
            node = self.get_node(node_key)
            if not node: continue
            
            # Process Relations
            for rel in node.get('relations', []):
                target = rel['targetId']
                if target not in visited:
                    visited.add(target)
                    new_path = path + [f"{node.get('name', node_key)} {rel['verb']} {target}"]
                    queue.append((target, new_path, certainty * 0.9))
            
            # Process Inheritance
            for group in node.get('groups', []):
                if group not in visited:
                    visited.add(group)
                    new_path = path + [f"{node.get('name', node_key)} is_a {group}"]
                    queue.append((group, new_path, certainty * 0.99))
        
        return None, 0

engine = PythonLogicEngine(db)
print("✅ Inference Core Loaded.")

In [ ]:
# @title 🔍 Step 3: Test Reasoning (Query Console)
SUBJECT = "Socrates" # @param {type:"string"}
PREDICATE = "mortal" # @param {type:"string"}

path, score = engine.find_path(SUBJECT, PREDICATE)

if path:
    print(f"✅ Logical Proof Found (Certainty: {score*100:.1f}%)")
    for i, step in enumerate(path):
        print(f"  Step {i+1}: {step}")
else:
    print("❌ No logical connection found in the database.")